Step 0 — Virtual Environment

In [1]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv

# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip

# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"

print("Virtual environment ready with vLLM installed!")

Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,265 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-d

Step 1 — Install the required packages

In [9]:
import subprocess, sys

# Pins mirrored from ../../PINS.md. Keep these two in sync (PINS.md wins).
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = ["/content/venv/bin/python", "-m", "pip", "install", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [10]:
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
# NOTE (2026-08-07, verified the hard way on a live T4): do NOT add a
# numpy>=2 pin here - vLLM 0.6.x requires numpy<2 and the install fails
# outright. CELL B as verified 2026-07-27 runs on the numpy vLLM chooses.
print("serving pins installed")

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
serving pins installed


Step 2 — Launch the AWQ server

In [39]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 37862, logging to /content/server.log


Step 3 — Check that the server becomes healthy

In [40]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 63s: http://localhost:8000/v1/models -> 200


Step 4 — Measure AWQ VRAM

In [13]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

11723 MiB


Step 5 — Measure AWQ tokens/s


In [42]:
import time
import requests
import statistics

URL = "http://localhost:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "messages": [
        {
            "role": "user",
            "content": "Explain how an inference server works in about 180 words."
        }
    ],
    "max_tokens": 200,
    "temperature": 0,
}

# warm-up
requests.post(URL, json=payload).raise_for_status()

rates = []

for i in range(5):
    start = time.perf_counter()

    r = requests.post(URL, json=payload)
    r.raise_for_status()
    data = r.json()

    elapsed = time.perf_counter() - start
    tokens = data["usage"]["completion_tokens"]
    rate = tokens / elapsed
    rates.append(rate)

    print(f"run {i+1}: {tokens} tokens, {elapsed:.2f}s, {rate:.1f} tokens/s")

awq_tokens_per_s = statistics.mean(rates)

print(f"\nAWQ average: {awq_tokens_per_s:.1f} tokens/s")

run 1: 188 tokens, 2.03s, 92.4 tokens/s
run 2: 188 tokens, 2.05s, 91.6 tokens/s
run 3: 188 tokens, 2.00s, 93.9 tokens/s
run 4: 188 tokens, 2.01s, 93.5 tokens/s
run 5: 188 tokens, 2.02s, 93.1 tokens/s

AWQ average: 92.9 tokens/s


Step 6 — Five-prompt quality spot check

In [20]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": p}],
        max_tokens=200
    )

    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content)
    print()

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests to execute model predictions and returning results to the client seamlessly and efficiently.

PROMPT: A user asks for the weather in Riyadh and the time ...
To find both the weather in Riyadh and the time in Tokyo, you would need to use two tool calls:

1. For the weather, you would make a tool call to "WeatherForecast".
2. For the time, you would make a tool call to "TimeIn" or "Time".

Here's an example of how you might structure the JSON for these two tool calls using a hypothetical JSON web API or a chatbot API provided by the service:

```json
{
    "tool": "WeatherForecast",
    "parameters": {
        "city": "Riyadh"
    },
    "_request_id": "123456789"
},
{
    "tool": "Time",
    "parameters": {
        "location": "Tokyo",
        "number_of_decimals": 2
    },
    "_request_id": "987654321"
}
```

In this example, I've included the API key plac

Step 7 — Shut down AWQ before loading fp16

In [16]:
import os, signal, time

if server.poll() is None:
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    server.wait(timeout=30)

print("AWQ server stopped")

AWQ server stopped


# ****Steps 8/9: change into fp16**

Step 10 — Run the function-calling smoke test

In [44]:
from openai import OpenAI

# Two tools the model may call. Shapes match the OpenAI tools schema.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

# The 3 canonical prompts. Each carries how many attempts (k) it gets and how many
# tool calls a correct answer makes. n = sum of k = 10.
#   two_tool:   needs BOTH tools (weather + calculator)   -> expect >= 1 call
#   single:     needs ONE tool                            -> expect >= 1 call
#   distractor: needs NO tool, must NOT call one          -> expect 0 calls
CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]


def _tool_calls_of(message) -> list:
    """Return the parsed tool_calls list on a response message, or []."""
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []


def _valid_call(call) -> bool:
    """A tool call is valid if it names a known function and its arguments
    parse as JSON with the required field present."""
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False


def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    """Run the smoke test. Returns a result dict with counts and the pass gate."""
    client = OpenAI(base_url=base_url, api_key="not-needed")

    total_attempts = 0
    valid_call_attempts = 0          # attempts that returned >=1 valid tool call
    distractor_attempts = 0
    distractor_call_free = 0         # distractor attempts that made NO tool call
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                # a "wants a call" prompt counts toward the 8/10 gate when it
                # returns at least one valid tool call
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                # the distractor counts toward the 8/10 gate when it correctly
                # makes NO tool call, and separately toward distractor compliance
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    # gate: >=8/10 correct behaviours AND distractor call-free in the majority
    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,       # 10
        "score": valid_call_attempts,           # correct behaviours, out of 10
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }

In [46]:
result = run_smoke(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ"
)

print(result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [24]:
fp16_result = result.copy()
print("Saved fp16:", fp16_result["score"], "/ 10")

Saved fp16: 10 / 10


In [47]:
import json

with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

print("saved smoke_result.json")

saved smoke_result.json


LAST STEP: Verify

In [50]:
import json, os, re


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) smoke result
    if not os.path.exists("smoke_result.json"):
        fail("smoke_result.json not found; write it in Cell 5")
    try:
        with open("smoke_result.json") as fh:
            result = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"smoke_result.json is not valid JSON: {exc}")

    for key in ("score", "total_attempts", "distractor_majority_clean", "passed"):
        if key not in result:
            fail(f"smoke_result.json missing key: {key}")

    score = result["score"]
    total = result["total_attempts"]
    if not isinstance(score, int) or not isinstance(total, int):
        fail("score and total_attempts must be integers")
    if total != 10:
        fail(f"total_attempts is {total}, the smoke test defines n=10")
    if score < 8:
        fail(f"smoke score {score}/10 is below the 8/10 gate")
    if not result["distractor_majority_clean"]:
        fail("distractor did not stay call-free in the majority; a model that "
             "always calls a tool fails the real consumer")
    if not result["passed"]:
        fail("smoke test reports passed=false")

    # 2) model-lock.md fully filled in
    if not os.path.exists("model-lock.md"):
        fail("model-lock.md not found")
    with open("model-lock.md") as fh:
        lock = fh.read()
    remaining = re.findall(r"FILL:", lock)
    if remaining:
        fail(f"model-lock.md has {len(remaining)} unfilled FILL: placeholders")
    # require a concrete model id line
    if not re.search(r"Model id:\s*\S+", lock):
        fail("model-lock.md has no concrete Model id")

    print(f"smoke score: {score}/{total}, distractor clean: "
          f"{result['distractor_majority_clean']}")
    print("model-lock.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)

smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
